# Synthetic Cohort Analysis — Digital Twin Engine

> ⚠️ **READ THIS FIRST.** Every figure this notebook produces comes from
> `dte.data.synthetic`, a generator with hand-authored assumptions. **No real patient
> contributed to any number below.** Accuracy here measures how well a model recovers the
> generator's assumptions, not anything about biology or about real people.
>
> See [`../DISCLAIMER.md`](../DISCLAIMER.md).

## What this notebook is for

It walks the analysis path an evaluation *would* follow, so the method can be reviewed before
any real data exists. Five sections:

1. Generate a synthetic cohort and inspect its structure
2. Run the edge signal pipeline, including the quality gate
3. Train the risk ensemble and read its explanations
4. **Run the fairness gate — and watch it block a release**
5. Simulate the Memory Anchoring Pipeline over a day

## Setup

```bash
pip install -r ../requirements-dev.txt
```


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dte.data.synthetic import generate_cohort, generate_eeg, generate_rr_intervals, generate_day
from dte.signals.eeg import extract_eeg_features
from dte.signals.hrv import extract_hrv_features
from dte.models.risk import RiskModel
from dte.models.decline import forecast_trajectory
from dte.fairness.subgroup import evaluate_fairness
from dte.fusion.detector import FusionInput, OpportunityDetector
from dte.policy.cue_selector import CueSelector, Response

pd.set_option('display.width', 120)
print('SYNTHETIC DATA ONLY — no real patient data is present or reachable from here.')


---
## 1. Generate a synthetic cohort

Note the two parameters that matter most: `underrepresented_fraction` makes one group small,
and `underrepresented_noise_multiplier` makes its signal noisier. Together they reproduce, in
miniature, the pattern the literature documents — accuracy falling on under-represented
populations (🔵 [14][19]). This exists so the fairness gate has something real to catch.


In [ ]:
cohort = generate_cohort(n=1200, underrepresented_noise_multiplier=2.5, seed=42)
X, y, attributes = cohort.to_arrays()

df = pd.DataFrame(X, columns=cohort.feature_names)
df['converter'] = y
for k, v in attributes.items():
    df[k] = v

print(f'cohort n={len(df)}  conversion rate={y.mean():.1%}')
df.head()


In [ ]:
# Group sizes — this is what determines whether the fairness gate CAN certify a subgroup.
for attr in ['race_ethnicity', 'primary_language', 'age_band', 'sex']:
    print(f'\n{attr}:')
    print(df[attr].value_counts().to_string())


### Does the injected signal look the way the generator claims?

Assumption 1 is that theta elevation tracks conversion risk. If that separation is not visible
here, the generator is broken — and if it *is* visible, remember that a model trained on this
data will 'confirm' an assumption we built in. That circularity is exactly why the
neural-channel ablation in [docs §7.6](../docs/07-validation-and-benchmarks.md#76-the-weakest-link)
matters.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat in zip(axes, ['theta_relative_power', 'theta_alpha_ratio', 'moca_total']):
    for label, grp in df.groupby('converter'):
        ax.hist(grp[feat], bins=30, alpha=0.6, label='converter' if label else 'stable')
    ax.set_title(feat); ax.set_xlabel(feat); ax.legend()
axes[0].set_ylabel('count')
fig.suptitle('SYNTHETIC — separation is by construction, not discovery', y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# Is the under-represented group actually noisier, as the generator claims?
noise_check = df.groupby('race_ethnicity')[['theta_relative_power', 'moca_total', 'gait_speed_ms']].std()
print('Within-group standard deviations (higher = noisier signal):')
noise_check.round(4)


---
## 2. The edge signal pipeline and its quality gate

The important behaviour: below the quality threshold the pipeline returns
`quality='insufficient'` and **no features at all**. It does not impute. A clinician sees
'insufficient data', not a score built on a guess
([docs §5.7](../docs/05-data-architecture.md#57-data-quality-contract)).


In [ ]:
rows = []
for rate in [0.05, 0.15, 0.30, 0.50, 0.70, 0.90]:
    raw, motion = generate_eeg(artefact_rate=rate, rng=np.random.default_rng(3))
    f = extract_eeg_features(raw, motion_mask=motion)
    rows.append({
        'artefact_rate': rate,
        'usable_epochs': round(f.usable_epoch_fraction, 3),
        'valid_channels': f.valid_channel_count,
        'quality': f.quality,
        'theta_alpha_ratio': round(f.theta_alpha_ratio, 3) if f.theta_alpha_ratio else None,
    })
pd.DataFrame(rows)


In [ ]:
# The agitation index: a hard veto, not a weighted input.
for label, ag in [('calm', False), ('agitated', True)]:
    h = extract_hrv_features(generate_rr_intervals(agitated=ag, rng=np.random.default_rng(7)))
    veto = h.agitation_index > 0.70
    print(f'{label:9} rmssd={h.rmssd_ms:6.1f}ms  lf/hf={h.lf_hf_ratio:5.2f}  '
          f'agitation={h.agitation_index:.3f}  vetoes_cue={veto}')


---
## 3. Train the risk ensemble

Tree ensembles rather than a deep network — for interpretability and for a CPU-only footprint
([ADR-0004](../docs/adr/0004-ensembles-over-deep-networks.md)).


In [ ]:
split = int(0.70 * len(y))
model = RiskModel().fit(X[:split], y[:split], cohort.feature_names)

y_pred = model.model.predict(X[split:])
proba = model.model.predict_proba(X[split:])[:, 1]

acc = (y_pred == y[split:]).mean()
sens = y_pred[y[split:] == 1].mean()
fpr = y_pred[y[split:] == 0].mean()
brier = ((proba - y[split:]) ** 2).mean()

print('🟡 ILLUSTRATIVE — synthetic data. These are NOT clinical results.')
print(f'  accuracy    = {acc:.1%}')
print(f'  sensitivity = {sens:.1%}   (benchmark B1.1 target >= 80%)')
print(f'  FPR         = {fpr:.1%}   (benchmark B1.2 target <= 20%)')
print(f'  Brier       = {brier:.4f} (benchmark B1.4 target <= 0.18)')


In [ ]:
# A score cannot exist without its reasoning (ADR-0006).
model.brier_score = float(brier)
out = model.predict(cohort.patients[3].features)
print(f'score={out.score:.3f}  band={out.band}\n')
for i, f in enumerate(out.top_features, 1):
    print(f'{i}. {f.name:<28} {f.contribution:+.4f}  {f.direction:<16} {f.plain_language}')
print('\nPatient-facing version:')
print(out.patient_facing_summary())


In [ ]:
# Trajectory forecasting refuses to imply certainty it does not have.
for label, times, values in [
    ('2 visits',  [0., 6.],                       [26., 24.]),
    ('4 visits',  [0., 6., 12., 18.],             [26., 25., 23.5, 22.]),
    ('12 visits', list(np.arange(0, 36, 3.)),     [26. - 0.18*t for t in np.arange(0, 36, 3.)]),
]:
    f = forecast_trajectory(times, values)
    width = f.ci_upper[-1] - f.ci_lower[-1]
    print(f'{label:10} suppressed={str(f.point_estimate_suppressed):5}  '
          f'confidence={f.confidence:.2f}  CI width at horizon={width:.1f}')
    print(f'           {f.note}\n')


---
## 4. The fairness gate

This is the section that matters most. The gate has **unconditional release-blocking
authority** ([ADR-0007](../docs/adr/0007-subgroup-fairness-gate.md)).

Watch two distinct failure modes: a measured accuracy gap, and a subgroup too small to certify.


In [ ]:
report = evaluate_fairness(y[split:], y_pred, {k: v[split:] for k, v in attributes.items()})
print(report.summary())
print()
fair_df = pd.DataFrame([s.to_dict() for s in report.subgroups])
fair_df


In [ ]:
if report.blocking_reasons:
    print('RELEASE BLOCKED:')
    for r in report.blocking_reasons: print(f'  - {r}')
    print('\nREMEDIATION:')
    for r in report.remediation: print(f'  - {r}')
else:
    print('Gate passed.')


In [ ]:
# How does the gap respond to injected noise?
gaps = []
for noise in [1.0, 1.5, 2.0, 2.5, 3.0, 4.0]:
    c = generate_cohort(n=1200, underrepresented_noise_multiplier=noise, seed=42)
    Xi, yi, ai = c.to_arrays()
    s = int(0.70 * len(yi))
    m = RiskModel().fit(Xi[:s], yi[:s], c.feature_names)
    rep = evaluate_fairness(yi[s:], m.model.predict(Xi[s:]), {k: v[s:] for k, v in ai.items()})
    gaps.append({'noise': noise, 'max_gap': rep.max_gap, 'group': rep.max_gap_group,
                 'passed': rep.passed})
gap_df = pd.DataFrame(gaps)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gap_df['noise'], gap_df['max_gap'], marker='o', label='max subgroup gap')
ax.axhline(0.10, color='red', linestyle='--', label='10 pt gate (blocks release)')
ax.set_xlabel('noise multiplier for under-represented group')
ax.set_ylabel('max subgroup accuracy gap')
ax.set_title('🟡 ILLUSTRATIVE — the gate activates as subgroup signal degrades')
ax.legend(); plt.tight_layout(); plt.show()
gap_df


---
## 5. Memory Anchoring Pipeline over a simulated day

Expect **silence to dominate**. The pipeline is precision-weighted: built to miss opportunities
rather than fire wrongly ([docs §4.2](../docs/04-memory-anchoring-pipeline.md#42-why-this-is-hard)).


In [ ]:
detector = OpportunityDetector()
selector = CueSelector(seed=42)
rng = np.random.default_rng(42)

records = []
delivered, last_cue_min = 0, None

for e in generate_day(seed=42, n_events=24):
    now_min = e.hour * 60 + e.minute
    theta_z = float(rng.normal(1.0, 0.9))
    agitation = 0.82 if e.agitated else float(np.clip(rng.normal(0.25, 0.08), 0, 1))
    d = detector.decide(FusionInput(
        place_class=e.place_class, place_significance=e.place_significance,
        hour_of_day=e.hour, agitation_index=agitation, theta_alpha_z=theta_z,
        eligible_content_count=e.eligible_content_count, signal_quality=0.82,
        minutes_since_last_cue=None if last_cue_min is None else now_min - last_cue_min,
        cues_delivered_today=delivered, consent_permits_cue=True))
    arm = None
    if d.deliver:
        sel = selector.select(e.place_class, e.hour)
        if sel:
            arm = sel.arm.value
            selector.update(sel, Response.ENGAGED)
            delivered += 1; last_cue_min = now_min
    records.append({'time': f'{e.hour:02d}:{e.minute:02d}', 'place': e.place_class,
                    'theta_z': round(theta_z, 2), 'agitation': round(agitation, 2),
                    'content': e.eligible_content_count,
                    'confidence': round(d.confidence, 3), 'deliver': d.deliver,
                    'arm': arm, 'reason': d.reason[:52]})

day_df = pd.DataFrame(records)
print(f'evaluations={len(day_df)}  delivered={delivered}  '
      f'silence rate={1 - delivered/len(day_df):.1%}')
day_df


In [ ]:
# Why was the system silent? Counting reasons is how the threshold gets calibrated.
print('Reasons for silence:')
print(day_df[~day_df['deliver']]['reason'].value_counts().to_string())


In [ ]:
suppressed = day_df[(day_df['confidence'] >= 0.62) & (~day_df['deliver'])]
print('Moments where confidence cleared the threshold but a hard gate still blocked delivery:')
print()
suppressed[['time', 'place', 'theta_z', 'agitation', 'confidence', 'reason']]


---
## What this notebook does not show

Being explicit, because a notebook full of plots reads as evidence:

- **Nothing about real patients.** Every figure is synthetic.
- **No validation of the theta-receptiveness assumption.** It is *built into the generator*, so
  the model recovering it proves nothing. See
  [docs §7.6](../docs/07-validation-and-benchmarks.md#76-the-weakest-link) for the ablation and
  sham-control designs that would actually test it.
- **No ambulatory reality.** Real EEG in a real home is harder than the generator's model of it.
- **No clinical outcomes.** Recall improvement, caregiver burden and hospitalisation cannot be
  simulated. They have to be measured.

### The single most useful experiment anyone could run

Disable the neural channel and see whether detection precision degrades. If it does not, the
EEG headset is adding cost, burden and privacy risk for nothing — and the architecture gets
simpler, cheaper and less invasive. **A negative result would be a good outcome, and it is
welcome as a pull request or issue.**
